In [1]:
import warnings
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd

from dpec.forecasting.backtest import score, walk_forward
from dpec.forecasting.model import FORMULAS, shared_model
from dpec.forecasting.ranges import coverage_table
from dpec.forecasting.targets import forecast_table

warnings.filterwarnings("ignore")  # statsmodels talks a lot; nothing it says changes these results

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
table = forecast_table(ROOT / "data/warehouse.duckdb")
pred = pd.read_parquet(ROOT / "data/forecasts/backtest.parquet")
model = pred[pred["forecaster"] == "shared_model"]
print(pred.shape, model["low"].notna().sum())


(18012, 17) 2810


In [2]:
print(coverage_table(model, by=("checkpoint", "season")).round(2).to_string(index=False))
first5 = model[(model["season"] == 2026) & (model["round"] <= 5)]
print(coverage_table(first5).round(2).to_string(index=False))


    checkpoint  season  rows  inside  width_pct
after_practice    2023   280    0.81       1.49
after_practice    2024   399    0.79       1.41
after_practice    2025   401    0.79       1.73
after_practice    2026   325    0.78       2.10
   after_quali    2023   280    0.84       1.45
   after_quali    2024   399    0.79       1.34
   after_quali    2025   401    0.80       1.57
   after_quali    2026   325    0.80       1.86
    checkpoint  rows  inside  width_pct
after_practice    95    0.68       1.86
   after_quali    95    0.76       1.71


In [3]:
by_forecaster = coverage_table(pred, by=("checkpoint", "forecaster")).round(2)
print(by_forecaster.sort_values(["checkpoint", "width_pct"]).to_string(index=False))


    checkpoint     forecaster  rows  inside  width_pct
after_practice   shared_model  1405    0.79       1.68
after_practice      team_form  1405    0.80       1.82
after_practice      last_race  1405    0.79       2.20
after_practice per_year_model  1405    0.79       2.43
after_practice      no_signal  1405    0.78       2.72
after_practice   signal_as_is  1405    0.79       2.99
   after_quali   shared_model  1405    0.80       1.55
   after_quali per_year_model  1405    0.80       1.78
   after_quali      team_form  1405    0.80       1.82
   after_quali   signal_as_is  1405    0.79       1.95
   after_quali      last_race  1405    0.79       2.20
   after_quali      no_signal  1405    0.78       2.72


1. Not a bug, the point of the method. 2026 is a new rules era: the model's errors got bigger, and the ranges are made from recent errors, so they grew. An honest forecaster says "I'm less sure" when it is.
2. In races 1–5 the 10-weekend window still holds mostly 2025 errors, which were smaller, so the ranges were too narrow for the new cars. They caught up a few races later. To a strategist: "for the first races of a new era, treat the range as too narrow."
3. Same honesty, narrower ranges: 1.55% wide after quali against 1.78% for the next best. A range is the honest price of the errors; smaller errors buy a narrower range, which is more useful on the pit wall.

In [4]:
c = model.dropna(subset=["p_top3"]).copy()
c["pace_rank"] = c.groupby(["checkpoint", "weekend"])["actual"].rank(method="first")
c["top3"] = c["pace_rank"] <= 3
c["drivers"] = c.groupby(["checkpoint", "weekend"])["actual"].transform("size")
c["said"] = pd.cut(c["p_top3"], [0, 0.05, 0.2, 0.4, 0.6, 0.8, 1], include_lowest=True)
reliability = c[c["checkpoint"] == "after_quali"].groupby("said", observed=True).agg(
    rows=("top3", "size"), said=("p_top3", "mean"), happened=("top3", "mean")
)
print(reliability.round(2).to_string())
brier = c.groupby("checkpoint").apply(
    lambda g: pd.Series(
        {
            "model": ((g["p_top3"] - g["top3"]) ** 2).mean(),
            "no_information": ((3 / g["drivers"] - g["top3"]) ** 2).mean(),
        }
    )
)
print(brier.round(4).to_string())

                rows  said  happened
said                                
(-0.001, 0.05]   783  0.01      0.00
(0.05, 0.2]      168  0.12      0.12
(0.2, 0.4]       229  0.31      0.21
(0.4, 0.6]       165  0.49      0.62
(0.6, 0.8]        58  0.67      0.79
(0.8, 1.0]         2  0.81      1.00
                 model  no_information
checkpoint                            
after_practice  0.0843          0.1315
after_quali     0.0769          0.1315


Yes: 0.077 after quali against 0.132 with no information, 42% better. After practice 0.084, still far better.
The chances are a bit too flat: drivers given 0.4–0.8 make the top 3 more often than said (0.49 → 0.62, 0.67 → 0.79), drivers given 0.2–0.4 less often (0.31 → 0.21). The favourites are under-rated.
A shared error moves everyone together and doesn't change the order. Drawn one by one, that shared part acts like extra noise between drivers, so the replays shuffle the order too much and the chances come out too flat. A fix to try LATER: draw a whole past weekend's errors together. Today it's written down as a known limit (model card, Part 11), not fixed.

### An ablation = take one input out, run the same backtest, see how much worse it gets

In [7]:
def without(term):
    """The shared model with one input taken out of both formulas."""
    formulas = {
        checkpoint: f.replace(f" + {term}", "").replace(f"{term} + ", "")
        for checkpoint, f in FORMULAS.items()
    }
    return partial(shared_model, formulas=formulas)

ablations = {
    "full model": shared_model,
    "without quali": without("quali_gap_pct"),
    "without team form": without("team_form"),
    "without driver vs team": without("driver_vs_team"),
    "without recency weights": partial(shared_model, halflife=None),
}
abl = score(walk_forward(table, ablations))
print(abl.pivot(index="forecaster", columns="checkpoint", values="mae_pct").sort_values("after_quali").round(3).to_string())

checkpoint               after_practice  after_quali
forecaster                                          
full model                        0.556        0.498
without driver vs team            0.567        0.503
without recency weights           0.553        0.513
without quali                     0.556        0.557
without team form                 0.871        0.593


Team form, by far: without it the error jumps to 0.871 after practice and 0.593 after quali. After quali the quali gap is second (0.557 without it). "Without quali" is the same as the full model after practice because the after-practice formula doesn't use it.
No. The half-life was chosen on the tune years (Lesson 14), where it was as good as equal weights after practice and better after quali; here it costs 0.003 after practice and saves 0.015 after quali. It's also what lets the model learn after a rules change.
Yes: it helps a little at both checkpoints, costs nothing, and it's the only input that knows two drivers in the same car aren't equally fast.